In [17]:
# https://judge.nitro-ai.org/competitions/nitro/pre-iaio-2026/4/view

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import emoji
import torch

In [2]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/preiaio-2026-4/train_data.csv")
test = pd.read_csv("/kaggle/input/datasets/abukanabek/preiaio-2026-4/test_data.csv")
subm = pd.read_csv("/kaggle/input/datasets/abukanabek/preiaio-2026-4/sample_output.csv")
support = pd.read_csv("/kaggle/input/datasets/abukanabek/preiaio-2026-4/support_examples.csv")

train.shape, test.shape, subm.shape, support.shape

((688, 2), (64, 2), (64, 3), (7, 2))

In [3]:
train.head()

,sentence_id,text
0,1000,"Alice looked all round the table, but there wa..."
1,1001,This time Alice waited patiently until it chos...
2,1002,Off with his head!’” “How dreadfully savage!”...
3,1003,"either the locks were too large, or the key wa..."
4,1004,"“I mean, what makes them so shiny?” Alice loo..."


In [4]:
test.head()

,sample_id,emoji_sequence
0,query_001,🕰️🕛🌑
1,query_002,👻📱💔
2,query_003,1️⃣🔵🌙
3,query_004,💀😂🤣
4,query_005,🍰🤏😎


In [5]:
support = pd.merge(support, train, on='sentence_id', how='left')
support

,emoji_sequence,sentence_id,text
0,🐈🧘🧶,1607,The cat sat on the mat.
1,❤️☕🌅,1630,I love drinking coffee in the morning.
2,🚀🌌☄️,1150,The rocket launched into space.
3,🙏📞📱,1527,Please call me on my cell phone.
4,🌧️🐈🐕,1101,It is raining cats and dogs.
5,⏳💰💸,1454,Time is money.
6,🌅🐦🐛,1241,The early bird catches the worm.


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [9]:
SLANG_MAP = {
    "🧢": "lie falsehood fake",
    "🐍": "traitor betrayal enemy",
    "👻": "ignored ghosted stopped texting",
    "☕": "gossip drama rumor",
    "🍰": "easy simple effortless",
    "🐐": "greatest best champion legend",
    "💀": "funny dying laughter",
    "🤡": "foolish stupid idiot",
    "🪜": "climb success",
    "🔥": "hot amazing lit",
    "🦵": "luck break",
    "👀": "look see agreement",
    "🐷": "impossible flying",
    "🌚": "mocking shade",

    # appended by ChatGPT
    "🫡": "respect salute acknowledged",
    "🗿": "stoic unbothered deadpan",
    "🫠": "melting overwhelmed embarrassed",
    "😭": "crying sad emotional",
    "😤": "annoyed determined fed up",
    "😮‍💨": "relief tired exhausted",
    "😎": "cool confident flex",
    "💅": "unbothered confident sassy",
    "🧍": "awkward standing there",
    "📌": "important remember pinned",
    "🧠": "smart think logic",
    "🗣️": "say speak opinion",
    "🫶": "love support appreciation",
    "✨": "sparkle magical aesthetic",
    "🧃": "tea gossip juicy",
    "🚩": "red flag warning toxic",
    "🟩": "green flag good sign",
    "🧬": "dna core identity coded",
    "📈": "growth improving leveling up",
    "🧍‍♂️": "npc background character",
    "🧍‍♀️": "npc background character",
    "🤝": "deal agreement teamwork",
    "🤨": "sus suspicious doubtful",
    "✅": "valid correct approved",
    "🫵": "you specifically callout",
    "🪦": "ended dead finished",
    "🚫": "no stop banned",
}

def process_emojis(text):
    for symbol, meaning in SLANG_MAP.items():
        if symbol in text:
            text += f" {meaning}"
    text = emoji.demojize(text, (' ', '')).strip()
    text = text.replace('_', ' ')
    return text

In [12]:
test['text'] = test['emoji_sequence'].apply(process_emojis)
test

,sample_id,emoji_sequence,text
0,query_001,🕰️🕛🌑,mantelpiece clock twelve o’clock new moon
1,query_002,👻📱💔,ghost mobile phone broken heart ignored ghoste...
2,query_003,1️⃣🔵🌙,keycap 1 blue circle crescent moon
3,query_004,💀😂🤣,skull face with tears of joy rolling on the fl...
4,query_005,🍰🤏😎,shortcake pinching hand smiling face with sung...
...,...,...,...
59,query_060,🚴‍♂️⛰️⬆️,man biking mountain up arrow
60,query_061,🐷✈️☁️,pig face airplane cloud impossible flying
61,query_062,☕🗣️🎭,hot beverage speaking head performing arts gos...
62,query_063,🦵💔🎭,leg broken heart performing arts luck break


In [14]:
corpus_embs = model.encode(train['text'].tolist(), convert_to_tensor=True)
query_embs = model.encode(test['text'].tolist(), convert_to_tensor=True)

corpus_embs.shape, query_embs.shape

(torch.Size([688, 384]), torch.Size([64, 384]))

In [28]:
chosen_indices = torch.matmul(query_embs, corpus_embs.T).argmax(dim=1)
answers = train['sentence_id'][chosen_indices.numpy()].values
answers

array([1232, 1321, 1192, 1404, 1012, 1400, 1213, 1002, 1402, 1262, 1156,
       1037, 1065, 1472, 1566, 1373, 1183, 1456, 1180, 1569, 1504, 1553,
       1681, 1433, 1487, 1510, 1637, 1636, 1317, 1425, 1681, 1520, 1292,
       1632, 1474, 1260, 1343, 1295, 1477, 1237, 1037, 1470, 1370, 1333,
       1092, 1600, 1472, 1070, 1218, 1017, 1128, 1374, 1116, 1205, 1245,
       1177, 1595, 1635, 1357, 1416, 1191, 1584, 1311, 1670])

In [30]:
subm['answer'] = answers

subm.to_csv("submission.csv", index=False)

subm.head()

,subtaskID,datapointID,answer
0,1,query_001,1232
1,1,query_002,1321
2,1,query_003,1192
3,1,query_004,1404
4,1,query_005,1012
